# Minimum Wage and Unemployment in Europe

## Project Overview
This project analyzes the relationship between minimum wage levels and unemployment rates across European countries from 2003 to 2025.

## Data Sources
- Minimum wage data: Eurostat API (earn_mw_cur)
- Unemployment rate data: Eurostat API (une_rt_a)
- Exchange rate data: Eurostat API (ert_bil_eur_a)

## Research Question
What is the relationship between minimum wage levels and unemployment rates in Europe?

## Main Finding
A statistically significant negative correlation exists between minimum wage and unemployment rate (slope = -0.003, R² = 0.12, p < 0.05).


## 1. Data Collection
Download minimum wage, unemployment rate, and exchange rate data from the Eurostat API.


In [ ]:
import logging
logging.basicConfig(level=logging.INFO, format="%(levelname)s - %(message)s")


In [ ]:
import requests

def test_eurostat_api(dataset_code, label):
    """Test if a Eurostat dataset is accessible via API"""
    url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset_code}?format=JSON&lang=EN"

    try:
        response = requests.get(url, timeout=30)
        if response.status_code == 200:
            data = response.json()
            dimensions = list(data['dimension'].keys())
            n_values = len(data['value'])
            print(f"OK {label} ({dataset_code})")
            print(f"   -> Successfully downloaded, {n_values} data points in total")
            print(f"   -> Dimensions: {dimensions}\n")
            return True
        else:
            print(f"FAILED {label} ({dataset_code}): HTTP {response.status_code}\n")
            return False
    except Exception as e:
        print(f"FAILED {label} ({dataset_code}): {e}\n")
        return False


print("=== Eurostat API Access Verification ===\n")

results = []
results.append(test_eurostat_api("earn_mw_cur", "Minimum Wage Data"))
results.append(test_eurostat_api("nama_10_gdp", "GDP Data"))
results.append(test_eurostat_api("une_rt_a",    "Unemployment Rate Data"))
results.append(test_eurostat_api("ert_bil_eur_a", "Exchange Rate Data"))

print(f"Verification result: {sum(results)}/4 datasets accessible")


In [ ]:
import pandas as pd
def eurostat_to_dataframe(dataset_code):
    url = f"https://ec.europa.eu/eurostat/api/dissemination/statistics/1.0/data/{dataset_code}?format=JSON&lang=EN"
    response = requests.get(url, timeout=30)
    data = response.json()

    dims = data['dimension']
    dim_names = list(dims.keys())

    dim_indices = []
    for dim in dim_names:
        categories = dims[dim]['category']
        index_map = {v: k for k, v in categories['index'].items()}
        dim_indices.append(index_map)

    rows = []
    for pos_str, value in data['value'].items():
        pos = int(pos_str)
        coords = []
        for i in range(len(dim_names) - 1, -1, -1):
            size = len(dim_indices[i])
            coords.insert(0, dim_indices[i][pos % size])
            pos //= size
        rows.append(coords + [value])

    df = pd.DataFrame(rows, columns=dim_names + ['value'])
    return df

mw_df = eurostat_to_dataframe("earn_mw_cur")
print(mw_df.head(10))
print("Shape:", mw_df.shape)


## 2. Data Cleaning
Clean and standardize the data, convert all currencies to EUR, and merge into a panel dataset.


In [ ]:

mw_clean = mw_df[mw_df['freq'] == 'S'].copy()
mw_clean = mw_clean[mw_clean['time'].str.endswith('S1')].copy()
mw_clean['year'] = mw_clean['time'].str[:4].astype(int)
mw_clean = mw_clean[['geo', 'year', 'value']].rename(columns={'value': 'min_wage'})

print(mw_clean.head(10))
print("Shape:", mw_clean.shape)


In [ ]:

une_df = eurostat_to_dataframe("une_rt_a")
print(une_df.head())
print("Dimensions:", une_df.columns.tolist())


In [ ]:
une_clean = une_df[
    (une_df['age'] == 'TOTAL') &
    (une_df['sex'] == 'T') &
    (une_df['unit'] == 'PC_ACT')
].copy()

une_clean['year'] = une_clean['time'].astype(int)
une_clean = une_clean[['geo', 'year', 'value']].rename(columns={'value': 'unemployment_rate'})

print(une_clean.head(10))
print("Shape:", une_clean.shape)


In [ ]:
print("age_all_value:", une_df['age'].unique())
print("sex_all_value:", une_df['sex'].unique())
print("unit_all_value:", une_df['unit'].unique())


In [ ]:
une_clean = une_df[
    (une_df['age'] == 'Y15-74') &
    (une_df['sex'] == 'T') &
    (une_df['unit'] == 'PC_ACT')
].copy()

une_clean['year'] = une_clean['time'].astype(int)
une_clean = une_clean[['geo', 'year', 'value']].rename(columns={'value': 'unemployment_rate'})

print(une_clean.head(10))
print("Shape:", une_clean.shape)


In [ ]:

panel = pd.merge(mw_clean, une_clean, on=['geo', 'year'], how='inner')

print(panel.head(10))
print("Shape:", panel.shape)
print("The_amount_of_country:", panel['geo'].nunique())
print("Year_range:", panel['year'].min(), "-", panel['year'].max())


In [ ]:

er_df = eurostat_to_dataframe("ert_bil_eur_a")
print(er_df.head())
print("dimension:", er_df.columns.tolist())
print("currency_value:", er_df['currency'].unique() if 'currency' in er_df.columns else "None")


In [ ]:

currencies_needed = ['CZK', 'HUF', 'MKD', 'RSD', 'TRY']

er_clean = er_df[
    (er_df['currency'].isin(currencies_needed)) &
    (er_df['statinfo'] == 'AVG') &
    (er_df['unit'] == 'NAC')
].copy()

er_clean['year'] = er_clean['time'].astype(int)
er_clean = er_clean[['currency', 'year', 'value']].rename(columns={'value': 'exchange_rate'})

print(er_clean.head(10))
print("Shape:", er_clean.shape)


In [ ]:

geo_to_currency = {
    'CZ': 'CZK',
    'HU': 'HUF',
    'MK': 'MKD',
    'RS': 'RSD',
    'TR': 'TRY'
}

panel['currency'] = panel['geo'].map(geo_to_currency)


panel_with_er = pd.merge(panel, er_clean, on=['currency', 'year'], how='left')

panel_with_er['min_wage_eur'] = panel_with_er.apply(
    lambda row: row['min_wage'] / row['exchange_rate']
    if pd.notna(row['exchange_rate'])
    else row['min_wage'],
    axis=1
)

panel_final = panel_with_er[['geo', 'year', 'min_wage_eur', 'unemployment_rate']].copy()

print(panel_final.head(10))
print("Shape:", panel_final.shape)


## 3. Visualization
Visualize the relationship between minimum wage and unemployment rate across countries and regions.


# Minimum Wage vs Unemployment Rate in Europe (2003–2025)

This scatter plot shows the relationship between minimum wages and unemployment rates across European countries between 2003 and 2025. While countries with higher minimum wages often exhibit higher unemployment rates, the data points are widely dispersed, suggesting that minimum wage levels alone do not explain differences in unemployment. Considerable variation exists even among countries with similar wage levels, indicating that additional economic and institutional factors are likely important.

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.cm as cm
import numpy as np

fig, ax = plt.subplots(figsize=(14, 8))

countries = sorted(panel_final['geo'].unique())
colors = cm.rainbow(np.linspace(0, 1, len(countries)))

for i, country in enumerate(countries):
    data = panel_final[panel_final['geo'] == country]
    ax.scatter(data['min_wage_eur'], data['unemployment_rate'],
               label=country, color=colors[i], alpha=0.7, s=40)

ax.set_xlabel('Minimum Wage (EUR)')
ax.set_ylabel('Unemployment Rate (%)')
ax.set_title('Minimum Wage vs Unemployment Rate in Europe (2003-2025)')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()


In [ ]:

selected = ['DE', 'FR', 'ES', 'PL', 'RO', 'BG']

fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 10))

for country in selected:
    data = panel_final[panel_final['geo'] == country].sort_values('year')
    ax1.plot(data['year'], data['min_wage_eur'], marker='o', label=country)
    ax2.plot(data['year'], data['unemployment_rate'], marker='o', label=country)

ax1.set_title('Minimum Wage Over Time')
ax1.set_ylabel('Minimum Wage (EUR)')
ax1.legend()

ax2.set_title('Unemployment Rate Over Time')
ax2.set_ylabel('Unemployment Rate (%)')
ax2.legend()

plt.tight_layout()
plt.show()


## 4. Regression Analysis

### 4.1 Simple Linear Regression

Run a simple linear regression to quantify the relationship between minimum wage and unemployment rate.



In [ ]:
from scipy import stats


slope, intercept, r_value, p_value, std_err = stats.linregress(
    panel_final['min_wage_eur'],
    panel_final['unemployment_rate']
)

print(f"slope (slope): {slope:.4f}")
print(f"intercept (intercept): {intercept:.4f}")
print(f"R² : {r_value**2:.4f}")
print(f"P_value (p-value): {p_value:.4f}")


In [ ]:
slope, intercept, r_value, p_value, std_err = stats.linregress(
    panel_final['min_wage_eur'],
    panel_final['unemployment_rate']
)

print(f"Slope: {slope:.4f}")
print(f"Intercept: {intercept:.4f}")
print(f"R²: {r_value**2:.4f}")
print(f"P-value: {p_value:.4f}")
print(f"\nInterpretation:")
print(f"A 1 EUR increase in minimum wage is associated with a {abs(slope):.4f} percentage point decrease in unemployment rate.")
print(f"The relationship is statistically significant (p < 0.05).")
print(f"Minimum wage explains {r_value**2*100:.1f}% of the variation in unemployment rate.")


# Minimum Wage vs Unemployment Rate with OLS Trendline

The OLS trendline summarizes the overall relationship between minimum wages and unemployment across all observations. Although the trendline indicates the average direction of the relationship, the substantial dispersion around the line suggests that the explanatory power of minimum wages alone is limited. The graph therefore motivates the inclusion of additional variables such as GDP in the regression analysis.

In [ ]:
import plotly.express as px

fig = px.scatter(
    panel_final,
    x='min_wage_eur',
    y='unemployment_rate',
    color='geo',
    hover_data=['geo', 'year', 'min_wage_eur', 'unemployment_rate'],
    title='Minimum Wage vs Unemployment Rate in Europe (2003-2025)',
    labels={
        'min_wage_eur': 'Minimum Wage (EUR)',
        'unemployment_rate': 'Unemployment Rate (%)',
        'geo': 'Country'
    },
    trendline='ols'
)

fig.show()


### 4.2 Multiple Regression with GDP as Control Variable

To check the robustness of our result, we add GDP as a control variable. Richer economies may both afford higher minimum wages and enjoy lower unemployment, so controlling for GDP helps isolate the minimum wage effect. We use log(GDP) to reduce scale differences across countries and mitigate multicollinearity.


In [ ]:
# Download GDP data
gdp_df = eurostat_to_dataframe("nama_10_gdp")
print(gdp_df.head())
print("Dimensions:", gdp_df.columns.tolist())
print("na_item values:", gdp_df['na_item'].unique())
print("unit values:", gdp_df['unit'].unique())


In [ ]:
gdp_clean = gdp_df[
    (gdp_df['na_item'] == 'B1G') &
    (gdp_df['unit'] == 'CP_MEUR')
].copy()

gdp_clean['year'] = gdp_clean['time'].astype(int)
gdp_clean = gdp_clean[['geo', 'year', 'value']].rename(columns={'value': 'gdp'})

print(gdp_clean.head(10))
print("Shape:", gdp_clean.shape)


In [ ]:
import statsmodels.api as sm

# Merge GDP into panel data
panel_gdp = pd.merge(panel_final, gdp_clean, on=['geo', 'year'], how='inner')
print("Shape after merge:", panel_gdp.shape)

# Multiple regression: unemployment ~ min_wage + GDP
X = panel_gdp[['min_wage_eur', 'gdp']]
X = sm.add_constant(X)
y = panel_gdp['unemployment_rate']

model = sm.OLS(y, X).fit()
print(model.summary())


In [ ]:
panel_gdp['log_gdp'] = np.log(panel_gdp['gdp'])

X = panel_gdp[['min_wage_eur', 'log_gdp']]
X = sm.add_constant(X)
y = panel_gdp['unemployment_rate']

model2 = sm.OLS(y, X).fit()
print(model2.summary())


In [ ]:
import matplotlib.pyplot as plt

e_y1 = sm.OLS(y, sm.add_constant(panel_gdp['log_gdp'])).fit().resid
e_x1 = sm.OLS(panel_gdp['min_wage_eur'], sm.add_constant(panel_gdp['log_gdp'])).fit().resid
e_y2 = sm.OLS(y, sm.add_constant(panel_gdp['min_wage_eur'])).fit().resid
e_x2 = sm.OLS(panel_gdp['log_gdp'], sm.add_constant(panel_gdp['min_wage_eur'])).fit().resid


region_map = {
    'Western Europe': ['BE', 'DE', 'FR', 'IE', 'LU', 'NL'],
    'Northern Europe': ['EE', 'LT', 'LV'],
    'Southern Europe': ['CY', 'EL', 'ES', 'HR', 'MT', 'PT', 'SI'],
    'Eastern Europe': ['BG', 'CZ', 'HU', 'MK', 'PL', 'RO', 'RS', 'SK'],
    'Other': ['ME', 'TR']
}
geo_to_region = {geo: region for region, geos in region_map.items() for geo in geos}
regions = panel_gdp['geo'].map(geo_to_region)

colors = {
    'Western Europe': '#4C72B0',
    'Northern Europe': '#55A868',
    'Southern Europe': '#C44E52',
    'Eastern Europe': '#DD8452',
    'Other': '#8172B3'
}

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))

for region, color in colors.items():
    mask = (regions == region).values
    ax1.scatter(e_x1[mask], e_y1[mask], alpha=0.45, s=18, color=color, label=region)
    ax2.scatter(e_x2[mask], e_y2[mask], alpha=0.45, s=18, color=color)

b1 = sm.OLS(e_y1, sm.add_constant(e_x1)).fit().params.iloc[1]
ax1.plot(e_x1, b1 * e_x1, color='black', linewidth=2)
ax1.set_xlabel('Minimum Wage (EUR), residualized')
ax1.set_ylabel('Unemployment Rate (%), residualized')
ax1.set_title('Effect of Minimum Wage\n(controlling for log GDP)')
ax1.legend(fontsize=9)

b2 = sm.OLS(e_y2, sm.add_constant(e_x2)).fit().params.iloc[1]
ax2.plot(e_x2, b2 * e_x2, color='black', linewidth=2)
ax2.set_xlabel('log(GDP), residualized')
ax2.set_ylabel('Unemployment Rate (%), residualized')
ax2.set_title('Effect of GDP\n(controlling for Minimum Wage)')

plt.tight_layout()
plt.show()



# Animated Bubble Chart (Bubble Size = GDP)

This animated visualization illustrates how the relationship between minimum wages and unemployment evolved over time. The size of each bubble represents a country's GDP, allowing comparison of both labor market outcomes and economic scale. The animation highlights substantial differences between countries and shows how many European economies experienced gradual increases in minimum wages throughout the observed period.

In [ ]:
plot_df = panel_gdp.copy()
plot_df = plot_df.sort_values('year')

all_combos = pd.MultiIndex.from_product(
    [plot_df['geo'].unique(), plot_df['year'].unique()],
    names=['geo', 'year']
).to_frame(index=False)

plot_full = all_combos.merge(plot_df, on=['geo', 'year'], how='left')


plot_full['gdp'] = plot_full['gdp'].fillna(0)
plot_full['min_wage_eur'] = plot_full['min_wage_eur'].fillna(0)
plot_full['unemployment_rate'] = plot_full['unemployment_rate'].fillna(0)

plot_full = plot_full.sort_values('year')
plot_full['year'] = plot_full['year'].astype(str)

fig = px.scatter(
    plot_full,
    x='min_wage_eur',
    y='unemployment_rate',
    color='geo',
    size='gdp',
    size_max=45,
    hover_name='geo',
    animation_frame='year',
    animation_group='geo',
    range_x=[0, plot_full['min_wage_eur'].max() * 1.05],
    range_y=[0, plot_full['unemployment_rate'].max() * 1.05],
    labels={
        'min_wage_eur': 'Minimum Wage (EUR)',
        'unemployment_rate': 'Unemployment Rate (%)',
        'geo': 'Country',
        'gdp': 'GDP (million EUR)'
    },
    title='Minimum Wage vs Unemployment, bubble size = GDP (2003-2025)'
)
fig.show()




# 5. Regional Analysis
Divide countries into geographical regions and analyze trends within each region.


## Minimum Wage in Europe Over Time

The choropleth map visualizes the geographical distribution of minimum wages across Europe from 2003 to 2025. A clear East–West divide can be observed, with Western and Northern European countries generally maintaining substantially higher minimum wages than Eastern European countries. Over time, many Eastern European countries exhibit strong wage growth, indicating partial convergence toward Western European wage levels.

In [ ]:
iso2_to_iso3 = {
    'AT': 'AUT',
    'BE': 'BEL',
    'BG': 'BGR',
    'CH': 'CHE',
    'CY': 'CYP',
    'CZ': 'CZE',
    'DE': 'DEU',
    'DK': 'DNK',
    'EE': 'EST',
    'EL': 'GRC',
    'ES': 'ESP',
    'FI': 'FIN',
    'FR': 'FRA',
    'HR': 'HRV',
    'HU': 'HUN',
    'IE': 'IRL',
    'IS': 'ISL',
    'IT': 'ITA',
    'LT': 'LTU',
    'LU': 'LUX',
    'LV': 'LVA',
    'MT': 'MLT',
    'NL': 'NLD',
    'NO': 'NOR',
    'PL': 'POL',
    'PT': 'PRT',
    'RO': 'ROU',
    'SE': 'SWE',
    'SI': 'SVN',
    'SK': 'SVK',
    'UK': 'GBR',
    'AL': 'ALB'
}


panel_map = panel_final.copy()
panel_map['iso3'] = panel_map['geo'].map(iso2_to_iso3)
panel_map = panel_map.dropna(subset=['iso3'])

panel_map = panel_map.sort_values('year')
fig = px.choropleth(
    panel_map,
    locations='iso3',
    color='min_wage_eur',
    hover_name='geo',
    hover_data={'min_wage_eur': True, 'unemployment_rate': True},
    animation_frame='year',
    scope='europe',
    color_continuous_scale='RdYlGn',
    title='Minimum Wage in Europe Over Time',
    labels={'min_wage_eur': 'Minimum Wage (EUR)'},
    width=900,
    height=600,
    range_color=[0, panel_map['min_wage_eur'].max()]
)

fig.show()


In [ ]:
fig.write_html("minimum_wage_map.html")
logging.info("Map saved to minimum_wage_map.html")


In [ ]:

region_map = {
    'Western Europe': ['BE', 'DE', 'FR', 'IE', 'LU', 'NL'],
    'Northern Europe': ['EE', 'LT', 'LV'],
    'Southern Europe': ['CY', 'EL', 'ES', 'HR', 'MT', 'PT', 'SI'],
    'Eastern Europe': ['BG', 'CZ', 'HU', 'MK', 'PL', 'RO', 'RS', 'SK'],
    'Other': ['ME', 'TR']
}


geo_to_region = {}
for region, countries in region_map.items():
    for country in countries:
        geo_to_region[country] = region

panel_final['region'] = panel_final['geo'].map(geo_to_region)

region_avg = panel_final.groupby(['region', 'year']).agg(
    min_wage_eur=('min_wage_eur', 'mean'),
    unemployment_rate=('unemployment_rate', 'mean')
).reset_index()

print(region_avg.head(10))


In [ ]:
fig1 = px.line(
    region_avg,
    x='year',
    y='min_wage_eur',
    color='region',
    title='Average Minimum Wage by Region Over Time',
    labels={'min_wage_eur': 'Minimum Wage (EUR)', 'year': 'Year', 'region': 'Region'}
)
fig1.show()

fig2 = px.line(
    region_avg,
    x='year',
    y='unemployment_rate',
    color='region',
    title='Average Unemployment Rate by Region Over Time',
    labels={'unemployment_rate': 'Unemployment Rate (%)', 'year': 'Year', 'region': 'Region'}
)
fig2.show()


In [ ]:

panel_final.to_csv('panel_data.csv', index=False)
region_avg.to_csv('region_data.csv', index=False)
logging.info("CSV files saved successfully")


## 6. Interactive Streamlit App
The interactive web application is built using Streamlit, allowing users to explore the data from different perspectives.

# Running the Streamlit Application

This notebook generates:

- `app.py`
- `panel_data.csv`
- `region_data.csv`

After running the notebook from start to finish, launch the Streamlit application with:

```bash
streamlit run app.py
```

Features:
- Filter by region and year range
- Interactive scatter plot with trendline
- Regional trend visualizations


In [ ]:
app_code = """
import streamlit as st
import pandas as pd
import plotly.express as px
from scipy import stats

panel = pd.read_csv('panel_data.csv')
region_avg = pd.read_csv('region_data.csv')

st.title('Minimum Wage and Unemployment in Europe')

st.sidebar.header('Filters')
selected_region = st.sidebar.selectbox(
    'Select Region',
    ['All'] + sorted(panel['region'].dropna().unique().tolist())
)
selected_years = st.sidebar.slider(
    'Select Year Range',
    int(panel['year'].min()),
    int(panel['year'].max()),
    (2010, 2023)
)

filtered = panel[
    (panel['year'] >= selected_years[0]) &
    (panel['year'] <= selected_years[1])
]
if selected_region != 'All':
    filtered = filtered[filtered['region'] == selected_region]

st.subheader('Minimum Wage vs Unemployment Rate')
fig1 = px.scatter(filtered, x='min_wage_eur', y='unemployment_rate',
    color='geo', hover_data=['geo', 'year'], trendline='ols')
st.plotly_chart(fig1)

st.subheader('Trends by Region')
fig2 = px.line(region_avg, x='year', y='min_wage_eur', color='region')
st.plotly_chart(fig2)
fig3 = px.line(region_avg, x='year', y='unemployment_rate', color='region')
st.plotly_chart(fig3)
"""

with open('app.py', 'w') as f:
    f.write(app_code)

logging.info("app.py created successfully")


## 7. Conclusion

Our analysis of 26 European countries (2003–2025) finds a statistically
significant **negative** correlation between minimum wage levels and
unemployment rates. In the simple linear regression, a €100 increase in
the monthly minimum wage is associated with a decrease of about 0.30
percentage points in the unemployment rate (p < 0.001), although minimum
wage alone explains only about 12% of the variation in unemployment
(R² ≈ 0.12).

This result is **robust to controlling for economic size**: after adding
log(GDP) as a control variable, the minimum wage coefficient remains
negative and highly significant (−0.0028, p < 0.001). The partial
regression plot confirms this relationship visually. Our findings are
consistent with Card & Krueger (1994) and Dube, Lester & Reich (2010),
who found no evidence that higher minimum wages increase unemployment.

The regional analysis reveals large and persistent gaps between Western
and Eastern Europe in minimum wage levels. However, the animated
visualization shows Eastern European minimum wages catching up rapidly
in recent years, while unemployment rates across Europe have generally
converged downward since the euro area crisis.

**Limitations.** Our results show correlation, not causation. Reverse
causality is possible: countries with stronger economies can afford both
higher minimum wages and lower unemployment. A causal analysis would
require methods such as difference-in-differences or instrumental
variables, which are beyond the scope of this project. In addition, our
panel does not control for country fixed effects, labor market
institutions, or differences in living costs across countries.
